In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# 힐베르트 포락선(Envelope) 분석 — 반복 이벤트의 주기를 드러내기

빠르게 진동하는 반송파(carrier)에 **느린 변조(modulation)** 가 실린 신호는 원 파형의 FFT 만으로는 변조 주기가 잘 드러나지 않는다. 힐베르트 변환으로 얻은 **해석신호의 크기**, 즉 포락선 $|x_a(t)| = \sqrt{x(t)^2 + \hat{x}(t)^2}$ 는 고주파 반송파를 벗겨 **느린 포락(엔진 firing, 충격 반복, 베어링 결함 임펄스)** 만을 남긴다. 포락선을 다시 FFT 로 분석하면 시간 파형에서는 잘 안 보이던 **반복 이벤트 주파수** 가 명확히 드러난다.

이 노트북의 목표

1. **실습 1.** Auto-rickshaw 엔진음 — 2-stroke 엔진의 firing 주기를 envelope FFT 로 확인한다.
2. **실습 2.** Knocking 엔진음 — 불규칙한 충격 신호에서 envelope 이 무엇을 드러내는지 관찰한다.
3. **실습 3.** 베어링 내륜 결함(CWRU DE_IR) — bandpass + envelope 로 **BPFI** 피크를 검출한다 (실전 진단 맥락).

## 공통 임포트와 데이터 로드

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import utils

try:
    import librosa
except ImportError:
    %pip install librosa
    import librosa

In [ ]:
# Auto-rickshaw / Knocking 엔진음
v_a, fs_a = librosa.load('./data/autorash.mp3', sr=None)
v_k, fs_k = librosa.load('./data/knocking.mp3',  sr=None)

# 두 신호 길이 맞춤 (Auto-rickshaw 를 Knocking 길이에 맞춰 잘라줌)
slice_length = len(v_k)
v_a = v_a[:slice_length]

T_a = len(v_a) / fs_a
t_a = np.arange(1/fs_a, T_a + 1/fs_a, 1/fs_a)
T_k = len(v_k) / fs_k
t_k = np.arange(1/fs_k, T_k + 1/fs_k, 1/fs_k)

print('Auto-rickshaw:', v_a.shape, 'fs =', fs_a)
print('Knocking     :', v_k.shape, 'fs =', fs_k)

## 실습 1. Auto-rickshaw 엔진음의 포락선

2-stroke Auto-rickshaw 엔진은 수백~수 kHz 의 공진(금속 진동·배기음)을 반송파로 하여 **수십 Hz 의 firing(폭발) 주기** 로 펄스가 반복된다. 원 파형의 FFT 는 공진 대역만 강조되어 firing 주기가 흐려지지만, 포락선을 FFT 하면 firing 주파수가 저주파(0–100 Hz) 영역에 뚜렷한 피크로 나타난다.

---

### 1-1. 시간 영역 포락선

In [ ]:
# Hilbert envelope of the raw waveform
v_a_env = utils.hilbert_envelope(v_a)

plt.figure(figsize=(8, 8))

plt.subplot(2, 1, 1)
plt.plot(t_a, v_a,      'C0', label='Auto-rickshaw')
plt.plot(t_a, v_a_env,  'C3', label='Envelope')
plt.xlabel('Time [sec]')
plt.ylabel('x(t)')
plt.title('Auto-rickshaw time-domain waveform with envelope')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(t_a, v_a,      'C0')
plt.plot(t_a, v_a_env,  'C3')
plt.xlim([0.60, 0.72])
plt.xlabel('Time [sec]')
plt.ylabel('x(t)')
plt.title('Auto-rickshaw waveform with envelope (zoom 0.60 - 0.72 s)')

plt.subplots_adjust(hspace=0.35)
plt.show()

### 1-2. 원 신호 vs 포락선의 주파수 분석 비교

In [ ]:
# Raw waveform spectrum
F_raw_a, A_raw_a = utils.fft(v_a     - v_a.mean(),     fs_a)
# Envelope spectrum (별도 이름으로 시간영역 v_a_env 를 덮어쓰지 않도록 주의)
F_env_a, A_env_a = utils.fft(v_a_env - v_a_env.mean(), fs_a)

plt.figure(figsize=(10, 10))

plt.subplot(4, 1, 1)
plt.plot(F_raw_a, A_raw_a, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Auto-rickshaw single-sided FFT (0 - 5000 Hz)')
plt.xlim([0, 5000])

plt.subplot(4, 1, 2)
plt.plot(F_raw_a, A_raw_a, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Auto-rickshaw single-sided FFT (zoom 0 - 100 Hz)')
plt.xlim([0, 100])

plt.subplot(4, 1, 3)
plt.plot(F_env_a, A_env_a, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope of Auto-rickshaw single-sided FFT (0 - 5000 Hz)')
plt.xlim([0, 5000])

plt.subplot(4, 1, 4)
plt.plot(F_env_a, A_env_a, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope of Auto-rickshaw FFT (zoom 0 - 100 Hz)')
plt.xlim([0, 100])

plt.subplots_adjust(hspace=0.6)
plt.show()

**관찰.** Auto-rickshaw 원 파형의 FFT 는 수백 Hz~수 kHz 의 공진 성분이 지배적이며 **firing 주기** 는 뚜렷하지 않다. 반면 포락선의 FFT 는 0–100 Hz 확대 구간에 **수십 Hz 대의 강한 피크와 배음** 이 나타나 엔진 폭발 반복 주파수가 명확하게 드러난다. 반송파의 FFT 는 주 공진만 보이지만, 포락선의 FFT 는 **반복 이벤트 주기**를 복조해 드러낸다는 것을 보여준다.

## 실습 2. Knocking 엔진음의 포락선

Knocking(노킹) 은 연소실의 이상 폭발이 불규칙한 **충격 펄스** 로 실리는 현상이다. 개별 펄스는 금속 공진을 여기시켜 고주파 파형으로 관찰되지만, **충격의 반복 패턴** 자체는 원 신호 FFT 에서 잘 안 보인다. 포락선을 통해 충격의 반복 구조를 주파수로 옮겨 본다.

---

### 2-1. 시간 영역 포락선

In [ ]:
v_k_env = utils.hilbert_envelope(v_k)

plt.figure(figsize=(8, 8))

plt.subplot(2, 1, 1)
plt.plot(t_k, v_k,      'C0', label='Knocking')
plt.plot(t_k, v_k_env,  'C3', label='Envelope')
plt.xlabel('Time [sec]')
plt.ylabel('x(t)')
plt.title('Knocking time-domain waveform with envelope')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(t_k, v_k,      'C0')
plt.plot(t_k, v_k_env,  'C3')
plt.xlim([0.60, 0.72])
plt.xlabel('Time [sec]')
plt.ylabel('x(t)')
plt.title('Knocking waveform with envelope (zoom 0.60 - 0.72 s)')

plt.subplots_adjust(hspace=0.35)
plt.show()

### 2-2. 원 신호 vs 포락선의 주파수 분석 비교

In [ ]:
F_raw_k, A_raw_k = utils.fft(v_k     - v_k.mean(),     fs_k)
F_env_k, A_env_k = utils.fft(v_k_env - v_k_env.mean(), fs_k)

plt.figure(figsize=(10, 10))

plt.subplot(4, 1, 1)
plt.plot(F_raw_k, A_raw_k, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Knocking single-sided FFT (0 - 5000 Hz)')
plt.xlim([0, 5000])

plt.subplot(4, 1, 2)
plt.plot(F_raw_k, A_raw_k, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Knocking single-sided FFT (zoom 0 - 100 Hz)')
plt.xlim([0, 100])

plt.subplot(4, 1, 3)
plt.plot(F_env_k, A_env_k, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope of Knocking single-sided FFT (0 - 5000 Hz)')
plt.xlim([0, 5000])

plt.subplot(4, 1, 4)
plt.plot(F_env_k, A_env_k, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope of Knocking FFT (zoom 0 - 100 Hz)')
plt.xlim([0, 100])

plt.subplots_adjust(hspace=0.6)
plt.show()

**관찰.** Knocking 원 파형의 FFT 는 수 kHz 의 공진 대역에 에너지가 모여 있어 충격의 **반복 구조** 는 직접 드러나지 않는다. 포락선 FFT 의 0–100 Hz 확대 구간에서는 반복 펄스의 주기에 해당하는 저주파 성분이 상대적으로 부각된다. Auto-rickshaw 와 달리 knocking 은 완전히 주기적이지 않기 때문에 피크가 얇고 선명하기보다 **넓게 퍼진 저주파 분포** 로 관찰된다. 이는 포락선 분석이 **완전 주기 신호뿐 아니라 반복 이벤트의 평균적 주기 구조** 도 드러낼 수 있음을 보여준다.

## 실습 3. 베어링 결함 신호의 포락선 — BPFI 검출

베어링 내륜 결함은 볼이 결함부를 지날 때마다 **짧은 임펄스** 를 발생시키고, 이 임펄스는 베어링 하우징의 **고주파 공진(수 kHz)** 을 여기시킨다. 즉 측정 신호는 *고주파 공진 반송파 × 저주파 결함 반복(BPFI) 변조* 형태다. 원 신호 FFT 에서는 공진 대역의 봉우리만 보이지만, 
**공진 대역 bandpass → 힐베르트 포락선 → FFT** 순서로 처리하면 저주파에서 **BPFI 와 그 배음** 이 드러난다 — 이것이 산업 현장에서 가장 널리 쓰이는 베어링 진단 파이프라인이다.

CWRU 벤치마크 데이터 `data/data_fault_DE_IR.csv` — Drive End (DE) 내륜 결함, rpm ≈ 1772, $f_s = 12000$ Hz — 로 확인한다. DE SKF6205 베어링의 BPFI 배수는 5.4152 이므로 **BPFI ≈ 5.4152 × (1772/60) ≈ 160 Hz** 가 기대된다.

---

### 3-1. 데이터 로드 및 BPFI 계산

In [ ]:
data = np.array(pd.read_csv('./data/data_fault_DE_IR.csv'))
fs_f = 12000
v_f  = data[:, 1]
N_f  = len(v_f)
t_f  = np.arange(N_f) / fs_f

rpm       = 1772
f_shaft   = rpm / 60.0
BPFI      = 5.4152 * f_shaft   # DE SKF6205 inner-race
print(f'N = {N_f}, duration = {N_f/fs_f:.2f} s')
print(f'Shaft frequency : {f_shaft:.3f} Hz')
print(f'BPFI            : {BPFI:.3f} Hz')

### 3-2. 원 시간파형과 원 스펙트럼 — 결함 반복 주기가 보이는가?

In [ ]:
F_raw_f, A_raw_f = utils.fft(v_f - v_f.mean(), fs_f)

plt.figure(figsize=(10, 7))

plt.subplot(3, 1, 1)
plt.plot(t_f, v_f, 'C0')
plt.xlim([0.0, 0.3])
plt.xlabel('Time [sec]');  plt.ylabel('x(t)')
plt.title('Bearing inner-race fault raw waveform (zoom 0 - 0.3 s)')

plt.subplot(3, 1, 2)
plt.plot(F_raw_f, A_raw_f, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Raw spectrum (0 - 6000 Hz)')
plt.xlim([0, 6000])

plt.subplot(3, 1, 3)
plt.plot(F_raw_f, A_raw_f, 'C0')
plt.axvline(BPFI, color='C3', linestyle='--', label=f'BPFI = {BPFI:.1f} Hz')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Raw spectrum (zoom 0 - 500 Hz)')
plt.xlim([0, 500])
plt.legend()

plt.subplots_adjust(hspace=0.6)
plt.show()

### 3-3. 공진 대역 Bandpass → 힐베르트 포락선

CWRU DE 내륜 결함 신호의 주요 공진 대역인 **3500–5500 Hz** 를 bandpass 로 추출한 뒤 힐베르트 포락선을 얻는다.

In [ ]:
# Resonance band bandpass + envelope
v_f_bp  = utils.filtering_zerophase(v_f, fs_f, 'band', f_low=3500, f_high=5500, order=4)
v_f_env = utils.hilbert_envelope(v_f_bp)

plt.figure(figsize=(10, 6))

plt.subplot(2, 1, 1)
plt.plot(t_f, v_f_bp, 'C0', label='Bandpass 3500-5500 Hz')
plt.plot(t_f, v_f_env, 'C3', label='Envelope')
plt.xlim([0.0, 0.3])
plt.xlabel('Time [sec]');  plt.ylabel('x(t)')
plt.title('Bandpass waveform with envelope (zoom 0 - 0.3 s)')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(t_f, v_f_env, 'C3')
plt.xlim([0.0, 0.1])
plt.xlabel('Time [sec]');  plt.ylabel('Envelope')
plt.title('Envelope only (zoom 0 - 0.1 s) - impulse repetition visible')

plt.subplots_adjust(hspace=0.5)
plt.show()

### 3-4. 포락선 스펙트럼 — BPFI 및 배음 검출

In [ ]:
F_env_f, A_env_f = utils.fft(v_f_env - v_f_env.mean(), fs_f)

plt.figure(figsize=(10, 6))

plt.subplot(2, 1, 1)
plt.plot(F_env_f, A_env_f, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope spectrum (0 - 1000 Hz)')
plt.xlim([0, 1000])

plt.subplot(2, 1, 2)
plt.plot(F_env_f, A_env_f, 'C3')
for k in range(1, 5):
    plt.axvline(k * BPFI, color='k', linestyle='--', alpha=0.6,
                label=(f'k x BPFI' if k == 1 else None))
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope spectrum with BPFI harmonics (zoom 0 - 500 Hz)')
plt.xlim([0, 500])
plt.legend()

plt.subplots_adjust(hspace=0.5)
plt.show()

**관찰.** 원 스펙트럼의 0–500 Hz 확대에서는 BPFI 근방이 잘 드러나지 않는다 — 결함 에너지가 **공진 대역(수 kHz)** 에 실려 있기 때문이다. 반면 **bandpass(3500–5500 Hz) → 포락선 → FFT** 결과의 0–500 Hz 구간에는 **약 160 Hz 부근 피크와 2×, 3×, 4× 배음** 이 일정한 간격으로 나타난다. 이것이 내륜 결함(BPFI)의 특징적 신호이며, 공진에 실린 임펄스의 반복이 포락선으로 복조되어 시각화된 결과다. 실습 1–2 에서 확인한 “포락선은 반복 이벤트의 주기를 드러낸다”는 원리가 **회전기계 결함 진단의 표준 파이프라인** 으로 그대로 이어진다.

---

---

## 정리

- 포락선은 **고주파 반송파를 벗기고 저주파 변조 성분만 남기는 복조 연산** 이다.
- `utils.hilbert_envelope(x)` 한 줄이면 힐베르트 해석신호의 크기를 얻을 수 있다.
- 베어링 진단의 표준 절차는 **공진 대역 bandpass → envelope → FFT** 이며, 결함 특성 주파수(BPFI/BPFO 등)와 그 배음을 확인한다.
